# Tokyo → London Phase 2b — Directional Asymmetry Exploratory
EA Main / UJ ShadowのQ5 UP/DOWNを1回だけ診断。Phase1/2の判定は変えません。
Plan SHA: e4021d03b5e5f88a3b91db8d3b2ad8dc3a330b6d。


In [ ]:
from pathlib import Path
import subprocess,sys
REPO=Path("/content/tokyo_london_phase2b_repo")
BRANCH="research/tokyo-london-phase2b-directional-asymmetry"
if not REPO.exists():
    subprocess.run(["git","clone","--branch",BRANCH,"https://github.com/TR-KJ/time-entry-portfolio-lab.git",str(REPO)],check=True)
IMPLEMENTATION_SHA=subprocess.check_output(["git","log","-1","--format=%H","--","src/research/tokyo_london_phase2b.py"],cwd=REPO,text=True).strip()
subprocess.run(["git","merge-base","--is-ancestor","e4021d03b5e5f88a3b91db8d3b2ad8dc3a330b6d",IMPLEMENTATION_SHA],cwd=REPO,check=True)
subprocess.run(["git","checkout","--detach",IMPLEMENTATION_SHA],cwd=REPO,check=True)
print("Implementation SHA:",IMPLEMENTATION_SHA)
subprocess.run([sys.executable,str(REPO/"tests/test_tokyo_london_phase2b.py")],check=True)


## 入力
Phase2日次assignment CSV（固定hash）と監査済みM1の16本を指定。Drive入力を使う場合だけmountを有効にします。


In [ ]:
MOUNT_DRIVE=False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
PHASE2_DAILY=Path("/content/tokyo_london_phase2_daily_assignment.csv")
DATA_ROOT=Path("/content/m1")
OUT=Path("/content/tokyo_london_phase2b")


In [ ]:
subprocess.run([sys.executable,str(REPO/"src/research/tokyo_london_phase2b.py"),"--phase2-daily",str(PHASE2_DAILY),"--data-root",str(DATA_ROOT),"--out",str(OUT),"--implementation-sha",IMPLEMENTATION_SHA],check=True)
subprocess.run([sys.executable,str(REPO/"tests/verify_tokyo_london_phase2b.py"),"--phase2-daily",str(PHASE2_DAILY),"--out",str(OUT)],check=True)


In [ ]:
import pandas as pd
from IPython.display import display
for name in ["primary_cells","pair_verdict","multiple_comparison","period_cells","asymmetry_summary","coverage","manual_audit","run_record"]:
    print(name)
    display(pd.read_csv(OUT/("tokyo_london_phase2b_"+name+".csv")))
print("CSV saved to",OUT)
print("Phase3 decision belongs to a human; no strategy was generated.")


## Drive保存（初期OFF）
本研究だけの新規日時付きフォルダへ保存。


In [ ]:
SAVE_TO_DRIVE=False
if SAVE_TO_DRIVE:
    from google.colab import drive
    import datetime,shutil
    drive.mount("/content/drive")
    destination=Path("/content/drive/MyDrive/time-entry-portfolio-lab/tokyo_london_phase2b")/datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    shutil.copytree(OUT,destination)
    print(destination)
